In [1]:
import os
import pandas as pd
import plotly.graph_objects as go
from typing import List, Optional, Dict
import numpy as np

def find_config_file(folder_path: str) -> Optional[str]:
    """
    Finds a configuration file (ending with .txt) within the 'configs' subfolder.
    """
    config_dir = os.path.join(folder_path, 'configs')
    if not os.path.isdir(config_dir):
        return None
    for item in os.listdir(config_dir):
        if item.endswith('.txt'):
            return os.path.join(config_dir, item)
    return None

def parse_config(file_path: str) -> Dict[str, str]:
    """
    Parses a 'key = value', 'key value', or 'key: value' configuration file into a dictionary.
    """
    params = {}
    try:
        with open(file_path, 'r') as f:
            for line in f:
                line = line.strip()
                if not line or line.startswith('#'):
                    continue
                
                separator = None
                if ':' in line:
                    separator = ':'
                elif '=' in line:
                    separator = '='

                if separator:
                    parts = line.split(separator, 1)
                else:
                    parts = line.split(None, 1)

                if len(parts) == 2:
                    key, value = parts
                    params[key.strip().lower()] = value.strip()
    except FileNotFoundError:
        print(f"Config file not found: {file_path}")
    except Exception as e:
        print(f"Error parsing config file {file_path}: {e}")
    return params

def find_timing_file(run_folder_path: str, sim_type: str) -> Optional[str]:
    """Finds the ...trace_matched_timing.csv file for a given run."""
    sim_output_dir = os.path.join(run_folder_path, sim_type.lower())
    if not os.path.isdir(sim_output_dir):
        return None
    for f in os.listdir(sim_output_dir):
        if 'trace_matched_timing.csv' in f:
            return os.path.join(sim_output_dir, f)
    return None

# --- Collective Info Parsing ---
def parse_collectives_log(log_path: str) -> Dict[str, Dict]:
    """Parses a duplicate_collectives.log file to extract signatures."""
    collective_info = {}
    try:
        with open(log_path, 'r') as f:
            lines = f.readlines()
            i = 0
            while i < len(lines):
                line = lines[i]
                workload_match = re.match(r'^Workload: (\S+)', line)
                if workload_match:
                    current_workload = workload_match.group(1)
                    # Look for signature on the next line
                    if (i + 1 < len(lines)) and (signature_match := re.match(r'^\s+Signature: \((.*)\)', lines[i+1])):
                        sig_content = signature_match.group(1).strip()
                        # Split signature into its three parts
                        parts = sig_content.rsplit(', ', 2)
                        if len(parts) == 3:
                            npu_tuples, comm_type, comm_size = parts
                            collective_info[current_workload] = {
                                'npu_tuples': npu_tuples.strip(),
                                'comm_type': comm_type.strip(),
                                'comm_size': comm_size.strip()
                            }
                i += 1
    except FileNotFoundError:
        print(f"Warning: Collectives log file not found at {log_path}")
    except Exception as e:
        print(f"Error parsing collectives log {log_path}: {e}")
    return collective_info

In [2]:
import re
import pandas as pd
from plotly.subplots import make_subplots

# --- Configuration ---
base_comparison_folders = [
    '/app/astra-sim/upc/output/comparison_run/FoldedClos/T5_Base_split',
    '/app/astra-sim/upc/output/comparison_run/FoldedClos/T5_Base_split_2',
    '/app/astra-sim/upc/output/comparison_run/FoldedClos/T5_Base_split_3',
    '/app/astra-sim/upc/output/comparison_run/FoldedClos/T5_Base_split_4'
]
comparison_plot_metric = 'avg'  # Can be 'avg' or 'max'

# --- Helper Functions ---
cc_modes = {0: "PFC", 1: "DCQCN", 3: "HPCC", 7: "TIMELY", 8: "DCTCP", 10: "HPCC-PINT"}
topo_idx_regex = re.compile(r'topology(\d+)(?:\.json)?$')

def parse_runtime(time_str: str) -> float:
    """Parses 'H:MM:SS.ffffff' into seconds."""
    if not time_str: return 0.0
    try:
        h, m, s = map(float, time_str.split(':'))
        return h * 3600 + m * 60 + s
    except (ValueError, IndexError):
        return 0.0

def process_simulation(folder: str, sim_type: str, summary_params: dict, collectives_data: dict, topo_index: int, run_name: str) -> Optional[dict]:
    """Processes a single simulation run, extracts timing data, and returns a result dictionary."""
    timing_file = find_timing_file(folder, sim_type)
    if not timing_file:
        return None

    try:
        df = pd.read_csv(timing_file)
        if sim_type == 'ns3' and 'node_name' in df.columns:
            df = df[df['node_name'] != 'dummy_node'].copy()

        time_col = 'callback_tick'
        if time_col not in df.columns:
            return None

        df = df[df[time_col] > 100].copy()
        elapsed_times = df[time_col].dropna()
        if elapsed_times.empty:
            return None

        workload_name = summary_params.get('collective', 'N/A').strip()
        collective_info = collectives_data.get(os.path.basename(workload_name), {})

        return {
            'workload': workload_name,
            'npu_count': summary_params.get('npus count', 'N/A'),
            'npu_tuples': collective_info.get('npu_tuples', 'N/A'),
            'comm_type': collective_info.get('comm_type', 'N/A'),
            'comm_size': collective_info.get('comm_size', 'N/A'),
            'topo_index': topo_index,
            'sim_type': sim_type.upper(),
            'run_name': run_name,
            'avg_time': elapsed_times.mean(),
            'max_time': elapsed_times.max(),
            'min_time': elapsed_times.min(),
            'std_dev': elapsed_times.std(),
            'execution_time': parse_runtime(summary_params.get('total runtime', '0:0:0.0')),
            'path': folder
        }
    except Exception as e:
        print(f"Error processing {sim_type} in {folder}: {e}")
        return None

# --- Data Collection Logic ---
all_run_folders = []
for base_folder in base_comparison_folders:
    for workload_folder in os.listdir(base_folder):
        workload_path = os.path.join(base_folder, workload_folder)
        if os.path.isdir(workload_path):
            all_run_folders.extend([os.path.join(workload_path, d) for d in os.listdir(workload_path) if os.path.isdir(os.path.join(workload_path, d))])

log_file_path = f'/app/astra-sim/upc/comparing_networks/workload/{os.path.basename(base_comparison_folders[0])}/duplicate_collectives.log' if base_comparison_folders else ''
collectives_data = parse_collectives_log(log_file_path)
if not collectives_data:
    print("Could not load collective signature data. Columns will be empty.")

comparison_results = []
for folder in sorted(all_run_folders):
    run_summary_path = os.path.join(folder, 'run_summary.txt')
    if not os.path.exists(run_summary_path):
        continue

    summary_params = parse_config(run_summary_path)
    
    # Process all potential simulation types in the folder
    sim_dirs = [d for d in os.listdir(folder) if os.path.isdir(os.path.join(folder, d)) and d in ['analytical_unaware', 'g2', 'ns3']]

    for sim_type in sim_dirs:
        topo_index = -1
        run_name = f"{sim_type.replace('_', ' ').title()}"

        if sim_type != 'analytical_unaware':
            topo_key = 'g2 topology file override' if sim_type == 'g2' else 'ns3 topology file override'
            topo_file = summary_params.get(topo_key, '')
            if 'all_paths' in topo_file: continue
            
            match = topo_idx_regex.search(topo_file)
            if not match: continue
            topo_index = int(match.group(1))

            if sim_type == 'ns3':
                ns3_config_file = find_config_file(folder)
                if ns3_config_file:
                    ns3_params = parse_config(ns3_config_file)
                    run_name = (
                        f"NS3 (cc:{cc_modes.get(int(ns3_params.get('cc_mode', -1)), 'N/A')}, "
                        f"win:{ns3_params.get('has_win', 'N/A')}, adapt:{ns3_params.get('var_win', 'N/A')}, "
                        f"buf:{ns3_params.get('buffer_size', 'N/A')}, size:{ns3_params.get('packet_payload_size', 'N/A')})"
                    )
            else: # G2
                run_name = "G2"

        result = process_simulation(folder, sim_type, summary_params, collectives_data, topo_index, run_name)
        if result:
            comparison_results.append(result)

# --- Plotting Logic ---


In [8]:

# --- Analysis and Plotting ---
if comparison_results:
    comp_df = pd.DataFrame(comparison_results)

    # Map for collective communication types
    comm_type_map = {
        "0": "ALL_REDUCE", "1": "REDUCE", "2": "ALL_GATHER", "3": "GATHER",
        "4": "SCATTER", "5": "BROADCAST", "6": "ALL_TO_ALL", "7": "REDUCE_SCATTER",
        "8": "REDUCE_SCATTER_BLOCK", "9": "BARRIER"
    }
    comp_df['comm_type_str'] = comp_df['comm_type'].astype(str).map(comm_type_map).fillna(comp_df['comm_type'])

    
    # Average metrics across topologies for each simulation run
    avg_topo_df = comp_df.groupby(['workload', 'sim_type', 'run_name']).agg(
        avg_time=('avg_time', 'mean'),
        max_time=('max_time', 'mean'),
        min_time=('min_time', 'mean'),
        comm_type_str=('comm_type_str', 'first'),
        comm_size=('comm_size', 'first'),
        npu_tuples=('npu_tuples', 'first'),
        path=('path', 'first') # Keep a path for later file access
    ).reset_index()

    def analyze_and_plot_divergence(metric_col, metric_name):
        print(f"--- Analyzing Divergence for: {metric_name} ---")
        
        divergence_data = []
        for wl, group in avg_topo_df.groupby('workload'):
            g2_runs = group[group['sim_type'] == 'G2']
            ns3_runs = group[group['sim_type'] == 'NS3']

            if g2_runs.empty or ns3_runs.empty:
                continue

            g2_time = g2_runs.iloc[0][metric_col]
            best_ns3_run = ns3_runs.loc[ns3_runs[metric_col].idxmin()]
            best_ns3_time = best_ns3_run[metric_col]

            if g2_time < 100 or best_ns3_time < 100:
                continue

            divergence = (g2_time - best_ns3_time)/g2_time
            info = g2_runs.iloc[0]
            divergence_data.append({
                'Workload': wl,
                'Divergence': divergence,
                'G2 Time': g2_time,
                'Best NS3 Time': best_ns3_time,
                'Best NS3 Run': best_ns3_run['run_name'],
                'comm_type': info['comm_type_str'],
                'comm_size': info['comm_size'],
                'npu_tuples': info['npu_tuples']
            })

        if not divergence_data:
            print(f"No divergent workloads found for {metric_name}.\n")
            return

        div_df = pd.DataFrame(divergence_data).sort_values(by='Divergence', ascending=False)
        top_10_workloads = div_df.head(10)

        print(f"\n--- Top 10 Most Divergent Workloads ({metric_name}) ---")
        with pd.option_context('display.float_format', '{:,.2f}'.format):
            display(top_10_workloads[['Workload', 'comm_type', 'comm_size', 'npu_tuples', 'G2 Time', 'Best NS3 Time', 'Divergence']])

        print(f"\n--- Generating Plots for Top 10 Divergent Workloads ({metric_name}) ---")
        for _, row in top_10_workloads.iterrows():
            wl = row['Workload']
            
            workload_df = comp_df[comp_df['workload'] == wl]
        
            group_df = workload_df

            g2_run = group_df[group_df['sim_type'] == 'G2'].iloc[0]
            ns3_runs_for_plot = group_df[group_df['sim_type'] == 'NS3'].sort_values(by=metric_col)
            best_ns3_run_for_plot = ns3_runs_for_plot.iloc[0]

            fig = make_subplots(rows=1, cols=2, subplot_titles=(f"Performance ({metric_name})", "Per-NPU Time Correlation"))

            # Bar plot
            fig.add_trace(go.Bar(x=ns3_runs_for_plot['run_name'], y=ns3_runs_for_plot[metric_col], name='NS3 Runs'), row=1, col=1)
            fig.add_hline(y=g2_run[metric_col], line_dash="dot", annotation_text=f"G2 Time", row=1, col=1)

            # Scatter plot
            g2_timing_file = find_timing_file(g2_run['path'], 'G2')
            ns3_timing_file = find_timing_file(best_ns3_run_for_plot['path'], 'NS3')
            if g2_timing_file and ns3_timing_file:
                try:
                    df_g2 = pd.read_csv(g2_timing_file)[['sys_id', 'callback_tick']].rename(columns={'callback_tick': 'g2_time'})
                    df_ns3 = pd.read_csv(ns3_timing_file)[['sys_id', 'callback_tick']].rename(columns={'callback_tick': 'ns3_time'})
                    merged_df = pd.merge(df_ns3, df_g2, on='sys_id')
                    merged_df = merged_df[(merged_df['ns3_time'] >= 100) & (merged_df['g2_time'] >= 100)]
                    
                    fig.add_trace(go.Scatter(x=merged_df['ns3_time'], y=merged_df['g2_time'], mode='markers', name='NPU Times'), row=1, col=2)
                    min_val = min(merged_df['ns3_time'].min(), merged_df['g2_time'].min())
                    max_val = max(merged_df['ns3_time'].max(), merged_df['g2_time'].max())
                    fig.add_trace(go.Scatter(x=[min_val, max_val], y=[min_val, max_val], mode='lines', name='y=x'), row=1, col=2)
                except Exception as e:
                    print(f"Could not create scatter plot for {wl}: {e}")
            
            plot_title = (f'<b>{wl}</b><br>'
                          f'Metric: {metric_name}<br>'
                          f'Comm: {row["comm_type"]}, Size: {row["comm_size"]}, NPUs: {row["npu_tuples"]}')

            fig.update_layout(title_text=plot_title, height=700, margin=dict(t=140))
            fig.show()
        print("\n" + "="*80 + "\n")


    # --- Run analysis for each metric ---
    analyze_and_plot_divergence('avg_time', 'Average Time')
    analyze_and_plot_divergence('max_time', 'Maximum Time')
    analyze_and_plot_divergence('min_time', 'Minimum Time')

else:
    print("No comparison results to process.")


--- Analyzing Divergence for: Average Time ---

--- Top 10 Most Divergent Workloads (Average Time) ---


,Workload,comm_type,comm_size,npu_tuples,G2 Time,Best NS3 Time,Divergence
247,T5_Base_split_3/0033_T5_Base_multiple_1_2_4_2_...,ALL_REDUCE,6291456,"((0, 2, 4, 6), (1, 3, 5, 7))","11,535,515,654.25","10,202,078,273.75",0.12
386,T5_Base_split_4/0064_T5_Base_multiple_2_1_4_2_...,ALL_REDUCE,12582912,"((8, 10, 12, 14), (9, 11, 13, 15))","23,071,031,297.50","20,508,630,148.12",0.11
248,T5_Base_split_3/0034_T5_Base_multiple_1_2_4_2_...,ALL_REDUCE,8388608,"((0, 2, 4, 6), (1, 3, 5, 7))","15,380,792,032.75","13,692,319,521.25",0.11
276,T5_Base_split_3/0062_T5_Base_multiple_2_1_4_2_...,ALL_REDUCE,12582912,"((0, 2, 4, 6), (1, 3, 5, 7))","23,071,031,297.50","20,607,847,521.25",0.11
168,T5_Base_split_2/0062_T5_Base_multiple_2_1_4_2_...,ALL_REDUCE,12582912,"((0, 2, 4, 6), (1, 3, 5, 7))","23,071,031,297.50","20,729,255,847.00",0.10
140,T5_Base_split_2/0034_T5_Base_multiple_1_2_4_2_...,ALL_REDUCE,8388608,"((0, 2, 4, 6), (1, 3, 5, 7))","15,380,792,032.75","13,865,978,658.00",0.10
139,T5_Base_split_2/0033_T5_Base_multiple_1_2_4_2_...,ALL_REDUCE,6291456,"((0, 2, 4, 6), (1, 3, 5, 7))","11,535,515,654.25","10,458,147,605.25",0.09
359,T5_Base_split_4/0037_T5_Base_multiple_1_2_4_2_...,REDUCE_SCATTER,67108864,"((8, 10, 12, 14), (9, 11, 13, 15))","51,808,091,532.00","47,168,623,351.12",0.09
362,T5_Base_split_4/0040_T5_Base_multiple_1_2_4_2_...,ALL_REDUCE,8388608,"((8, 10, 12, 14), (9, 11, 13, 15))","15,380,792,032.75","14,105,868,128.00",0.08
360,T5_Base_split_4/0038_T5_Base_multiple_1_2_4_2_...,ALL_GATHER,16777216,"((8, 10, 12, 14), (9, 11, 13, 15))","51,808,068,992.25","47,596,254,962.75",0.08



--- Generating Plots for Top 10 Divergent Workloads (Average Time) ---




--- Analyzing Divergence for: Maximum Time ---

--- Top 10 Most Divergent Workloads (Maximum Time) ---


,Workload,comm_type,comm_size,npu_tuples,G2 Time,Best NS3 Time,Divergence
359,T5_Base_split_4/0037_T5_Base_multiple_1_2_4_2_...,REDUCE_SCATTER,67108864,"((8, 10, 12, 14), (9, 11, 13, 15))","64,760,108,856.00","56,775,087,536.00",0.12
138,T5_Base_split_2/0032_T5_Base_multiple_1_2_4_2_...,ALL_GATHER,16777216,"((0, 2, 4, 6), (1, 3, 5, 7))","64,760,086,310.00","58,403,098,020.00",0.10
137,T5_Base_split_2/0031_T5_Base_multiple_1_2_4_2_...,REDUCE_SCATTER,67108864,"((0, 2, 4, 6), (1, 3, 5, 7))","64,760,108,856.00","58,434,941,112.00",0.10
247,T5_Base_split_3/0033_T5_Base_multiple_1_2_4_2_...,ALL_REDUCE,6291456,"((0, 2, 4, 6), (1, 3, 5, 7))","11,889,676,157.00","10,796,994,020.00",0.09
168,T5_Base_split_2/0062_T5_Base_multiple_2_1_4_2_...,ALL_REDUCE,12582912,"((0, 2, 4, 6), (1, 3, 5, 7))","23,779,352,308.00","21,754,322,847.00",0.09
248,T5_Base_split_3/0034_T5_Base_multiple_1_2_4_2_...,ALL_REDUCE,8388608,"((0, 2, 4, 6), (1, 3, 5, 7))","15,853,009,246.00","14,511,670,020.00",0.08
200,T5_Base_split_2/0094_T5_Base_multiple_2_4_2_1_...,ALL_GATHER,8388608,"((0, 2, 4, 6), (1, 3, 5, 7), (8, 10, 12, 14), ...","32,380,125,671.00","29,660,398,030.00",0.08
201,T5_Base_split_2/0095_T5_Base_multiple_2_4_2_1_...,REDUCE_SCATTER,33554432,"((0, 2, 4, 6), (1, 3, 5, 7), (8, 10, 12, 14), ...","32,380,136,939.00","29,660,413,054.00",0.08
416,T5_Base_split_4/0094_T5_Base_multiple_2_4_2_1_...,ALL_GATHER,8388608,"((0, 2, 4, 6), (1, 3, 5, 7), (8, 10, 12, 14), ...","32,380,125,672.00","29,696,301,519.00",0.08
360,T5_Base_split_4/0038_T5_Base_multiple_1_2_4_2_...,ALL_GATHER,16777216,"((8, 10, 12, 14), (9, 11, 13, 15))","64,760,086,310.00","59,461,477,782.00",0.08



--- Generating Plots for Top 10 Divergent Workloads (Maximum Time) ---




--- Analyzing Divergence for: Minimum Time ---

--- Top 10 Most Divergent Workloads (Minimum Time) ---


,Workload,comm_type,comm_size,npu_tuples,G2 Time,Best NS3 Time,Divergence
386,T5_Base_split_4/0064_T5_Base_multiple_2_1_4_2_...,ALL_REDUCE,12582912,"((8, 10, 12, 14), (9, 11, 13, 15))","22,059,142,315.00","15,688,145,519.00",0.29
115,T5_Base_split_2/0007_T5_Base_multiple_2_2_2_2_...,ALL_GATHER,16777216,"((8, 10), (9, 11), (12, 14), (13, 15))","17,269,356,020.00","12,703,117,151.00",0.26
116,T5_Base_split_2/0008_T5_Base_multiple_2_2_2_2_...,REDUCE_SCATTER,33554432,"((8, 10), (9, 11), (12, 14), (13, 15))","17,269,371,044.00","12,703,132,175.00",0.26
362,T5_Base_split_4/0040_T5_Base_multiple_1_2_4_2_...,ALL_REDUCE,8388608,"((8, 10, 12, 14), (9, 11, 13, 15))","14,706,194,797.00","11,193,471,824.00",0.24
139,T5_Base_split_2/0033_T5_Base_multiple_1_2_4_2_...,ALL_REDUCE,6291456,"((0, 2, 4, 6), (1, 3, 5, 7))","11,029,571,167.00","9,312,632,864.00",0.16
140,T5_Base_split_2/0034_T5_Base_multiple_1_2_4_2_...,ALL_REDUCE,8388608,"((0, 2, 4, 6), (1, 3, 5, 7))","14,706,194,797.00","12,429,279,796.00",0.15
276,T5_Base_split_3/0062_T5_Base_multiple_2_1_4_2_...,ALL_REDUCE,12582912,"((0, 2, 4, 6), (1, 3, 5, 7))","22,059,142,315.00","18,650,760,020.00",0.15
168,T5_Base_split_2/0062_T5_Base_multiple_2_1_4_2_...,ALL_REDUCE,12582912,"((0, 2, 4, 6), (1, 3, 5, 7))","22,059,142,315.00","18,683,510,847.00",0.15
248,T5_Base_split_3/0034_T5_Base_multiple_1_2_4_2_...,ALL_REDUCE,8388608,"((0, 2, 4, 6), (1, 3, 5, 7))","14,706,194,797.00","12,500,238,020.00",0.15
247,T5_Base_split_3/0033_T5_Base_multiple_1_2_4_2_...,ALL_REDUCE,6291456,"((0, 2, 4, 6), (1, 3, 5, 7))","11,029,571,167.00","9,413,484,030.00",0.15



--- Generating Plots for Top 10 Divergent Workloads (Minimum Time) ---


In [7]:
comp_df

,workload,npu_count,npu_tuples,comm_type,comm_size,topo_index,sim_type,run_name,avg_time,max_time,min_time,std_dev,execution_time,path,comm_type_str
0,T5_Base_split/0000_T5_Base_multiple_2_2_2_2_1_...,16,"((0, 2), (1, 3), (4, 6), (5, 7))",2,16777216,-1,ANALYTICAL_UNAWARE,Analytical Unaware,7.450581e+09,7450580616,7450580616,0.000000e+00,0.934860,/app/astra-sim/upc/output/comparison_run/Folde...,ALL_GATHER
1,T5_Base_split/0000_T5_Base_multiple_2_2_2_2_1_...,16,"((0, 2), (1, 3), (4, 6), (5, 7))",2,16777216,1,G2,G2,1.511069e+10,17269356020,8634678020,3.997079e+09,0.631840,/app/astra-sim/upc/output/comparison_run/Folde...,ALL_GATHER
2,T5_Base_split/0000_T5_Base_multiple_2_2_2_2_1_...,16,"((0, 2), (1, 3), (4, 6), (5, 7))",2,16777216,1,NS3,"NS3 (cc:DCTCP, win:1, adapt:1, buf:1, size:1500)",1.520187e+10,19612678317,10736738082,3.881208e+09,10.196461,/app/astra-sim/upc/output/comparison_run/Folde...,ALL_GATHER
3,T5_Base_split/0000_T5_Base_multiple_2_2_2_2_1_...,16,"((0, 2), (1, 3), (4, 6), (5, 7))",2,16777216,1,NS3,"NS3 (cc:DCQCN, win:1, adapt:1, buf:1, size:1500)",1.771885e+10,17719422020,17718220020,4.395192e+05,22.902080,/app/astra-sim/upc/output/comparison_run/Folde...,ALL_GATHER
4,T5_Base_split/0000_T5_Base_multiple_2_2_2_2_1_...,16,"((0, 2), (1, 3), (4, 6), (5, 7))",2,16777216,1,NS3,"NS3 (cc:PFC, win:1, adapt:1, buf:1, size:1500)",1.771885e+10,17719446020,17718220020,4.499117e+05,10.042329,/app/astra-sim/upc/output/comparison_run/Folde...,ALL_GATHER
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
2143,T5_Base_split_4/0109_T5_Base_multiple_8_2_1_1_...,16,"((0, 1, 2, 3, 4, 5, 6, 7), (8, 9, 10, 11, 12, ...",0,6291456,-1,ANALYTICAL_UNAWARE,Analytical Unaware,4.889449e+09,4889448574,4889448574,0.000000e+00,0.995093,/app/astra-sim/upc/output/comparison_run/Folde...,ALL_REDUCE
2144,T5_Base_split_4/0109_T5_Base_multiple_8_2_1_1_...,16,"((0, 1, 2, 3, 4, 5, 6, 7), (8, 9, 10, 11, 12, ...",0,6291456,4,G2,G2,5.666729e+09,5666729064,5666729064,0.000000e+00,0.701001,/app/astra-sim/upc/output/comparison_run/Folde...,ALL_REDUCE
2145,T5_Base_split_4/0109_T5_Base_multiple_8_2_1_1_...,16,"((0, 1, 2, 3, 4, 5, 6, 7), (8, 9, 10, 11, 12, ...",0,6291456,4,NS3,"NS3 (cc:DCTCP, win:1, adapt:1, buf:1, size:1500)",6.870163e+09,7018135764,6711469058,9.076013e+07,12.048677,/app/astra-sim/upc/output/comparison_run/Folde...,ALL_REDUCE
2146,T5_Base_split_4/0109_T5_Base_multiple_8_2_1_1_...,16,"((0, 1, 2, 3, 4, 5, 6, 7), (8, 9, 10, 11, 12, ...",0,6291456,4,NS3,"NS3 (cc:DCQCN, win:1, adapt:1, buf:1, size:1500)",6.901230e+09,7183670658,6772706593,1.129925e+08,21.979729,/app/astra-sim/upc/output/comparison_run/Folde...,ALL_REDUCE


In [6]:
# --- Detailed Difference and Correlation Analysis ---

if 'comp_df' in locals() and not comp_df.empty:
    # Pivot the table to have sim types as columns for easier comparison
    pivot_df = avg_topo_df.pivot_table(
        index=['workload', 'comm_type_str', 'comm_size', 'npu_tuples'],
        columns='run_name',
        values='avg_time'
    ).reset_index()

    # Isolate G2 and NS3 columns
    g2_col = 'G2'
    ns3_cols = [col for col in pivot_df.columns if col.startswith('NS3')]

    if g2_col in pivot_df.columns and ns3_cols:
        # Create a new DataFrame to store the detailed differences
        diff_data = []

        # Iterate over each workload
        for _, row in pivot_df.iterrows():
            g2_time = row[g2_col]
            if pd.isna(g2_time):
                continue

            # For each NS3 run, calculate the difference
            for ns3_run_name in ns3_cols:
                ns3_time = row[ns3_run_name]
                if pd.isna(ns3_time):
                    continue
                
                # Calculate absolute and relative differences
                abs_diff = ns3_time - g2_time
                rel_diff = (abs_diff / g2_time) * 100 if g2_time != 0 else 0

                diff_data.append({
                    'Workload': row['workload'],
                    'Comm Type': row['comm_type_str'],
                    'Comm Size': row['comm_size'],
                    'NPU Tuples': row['npu_tuples'],
                    'NS3 Run': ns3_run_name,
                    'G2 Time': g2_time,
                    'NS3 Time': ns3_time,
                    'Absolute Diff (NS3 - G2)': abs_diff,
                    'Relative Diff (%)': rel_diff
                })

        if diff_data:
            # Create the initial difference dataframe
            diff_df_all = pd.DataFrame(diff_data)

            # --- Filter for the specific NS3 DCTCP run ---
            target_ns3_run = 'NS3 (cc:DCTCP, win:1, adapt:1, buf:1, size:1500)'
            print(f"--- Filtering results for NS3 run: {target_ns3_run} ---")
            diff_df = diff_df_all[diff_df_all['NS3 Run'] == target_ns3_run].copy()

            if diff_df.empty:
                print("\nNo data found for the specified NS3 run. Please check the run name and the data.")
            else:
                # --- Display Difference Table ---
                print("\n--- G2 vs. NS3 (DCTCP) Time Difference Analysis ---")
                with pd.option_context('display.max_rows', 20, 'display.float_format', '{:,.2f}'.format):
                    display(diff_df)

                # --- Correlation Analysis ---
                print("\n--- Correlation Analysis (for DCTCP run) ---")
                
                # Function to parse communication size string (e.g., '1024 B', '2 KB', '4096') into bytes
                def parse_comm_size_to_bytes(size_str):
                    if not isinstance(size_str, str):
                        return np.nan
                    size_str = size_str.strip().lower()
                    parts = size_str.split()
                    
                    try:
                        if len(parts) == 1:
                            return float(parts[0]) # Assume bytes if no unit
                        elif len(parts) == 2:
                            val = float(parts[0])
                            unit = parts[1]
                            if unit == 'b':
                                return val
                            elif unit == 'kb':
                                return val * 1024
                            elif unit == 'mb':
                                return val * 1024**2
                            elif unit == 'gb':
                                return val * 1024**3
                            else:
                                return np.nan
                        else:
                            return np.nan
                    except (ValueError, TypeError):
                        return np.nan

                # Prepare data for correlation
                corr_df = diff_df.copy()
                
                # Convert Comm Type to numeric using one-hot encoding
                corr_df = pd.get_dummies(corr_df, columns=['Comm Type'], prefix='Comm')

                corr_df['comm_size_bytes'] = corr_df['Comm Size'].apply(parse_comm_size_to_bytes)
                
                # Extract NPU count from the tuple string
                corr_df['npu_count'] = corr_df['NPU Tuples'].apply(lambda x: len(eval(x)) if isinstance(x, str) and x.startswith('(') else np.nan)
                
                # Select only numeric columns for correlation matrix
                numeric_cols = corr_df.select_dtypes(include=np.number)

                if not numeric_cols.empty:
                    correlation_matrix = numeric_cols.corr()
                    
                    # Display correlation with the relative difference
                    print("Correlation with 'Relative Diff (%)':")
                    display(correlation_matrix[['Relative Diff (%)']].sort_values(by='Relative Diff (%)', ascending=False))

                    # Plotting correlation
                    fig = go.Figure(data=go.Heatmap(
                        z=correlation_matrix.values,
                        x=correlation_matrix.columns,
                        y=correlation_matrix.columns,
                        colorscale='RdBu_r',
                        zmin=-1,
                        zmax=1,
                        text=correlation_matrix.round(2).values,
                        texttemplate="%{text}"
                    ))
                    fig.update_layout(
                        title='Correlation Matrix (for DCTCP run)',
                        height=600
                    )
                    fig.show()
                else:
                    print("Not enough numeric data to compute correlations for the filtered data.")

        else:
            print("No difference data could be generated.")
    else:
        print("Could not find 'G2' and/or NS3 runs in the processed data to compare.")
else:
    print("Comparison results DataFrame ('comp_df') not found. Please run the preceding cells.")

# --- Communication Type Impact Analysis (for DCTCP run) ---
import plotly.express as px

if 'diff_df' in locals() and not diff_df.empty:
    print("\n--- Analyzing the Impact of Communication Type (for DCTCP run) ---")
    
    # Ensure the 'Comm Type' column exists
    if 'Comm Type' in diff_df.columns:
        
        # --- Box Plot for Relative Difference by Communication Type ---
        print("\nThis plot shows how the percentage difference between NS3 (DCTCP) and G2 varies for each communication type.")
        fig1 = px.box(diff_df, 
                      x='Comm Type', 
                      y='Relative Diff (%)', 
                      title='Relative Difference (NS3-DCTCP vs. G2) by Communication Type',
                      points="all")
        fig1.update_xaxes(categoryorder='total descending')
        fig1.show()

        # --- Box Plot for G2 Time by Communication Type ---
        print("\nThis plot shows the distribution of G2 simulation times for each communication type.")
        fig2 = px.box(diff_df, 
                      x='Comm Type', 
                      y='G2 Time', 
                      title='G2 Simulation Time by Communication Type',
                      points="all")
        fig2.update_xaxes(categoryorder='total descending')
        fig2.show()

        # --- Box Plot for NS3 Time by Communication Type ---
        print("\nThis plot shows the distribution of NS3 (DCTCP) simulation times for each communication type.")
        fig3 = px.box(diff_df, 
                      x='Comm Type', 
                      y='NS3 Time', 
                      title='NS3 (DCTCP) Simulation Time by Communication Type',
                      points="all")
        fig3.update_xaxes(categoryorder='total descending')
        fig3.show()

        # --- Grouped Analysis Table ---
        print("\n--- Average Metrics by Communication Type (for DCTCP run) ---")
        comm_type_analysis = diff_df.groupby('Comm Type').agg(
            avg_g2_time=('G2 Time', 'mean'),
            avg_ns3_time=('NS3 Time', 'mean'),
            avg_relative_diff=('Relative Diff (%)', 'mean'),
            count=('Workload', 'nunique')
        ).sort_values(by='avg_relative_diff', ascending=False)
        
        with pd.option_context('display.float_format', '{:,.2f}'.format):
            display(comm_type_analysis)

    else:
        print("Column 'Comm Type' not found in the filtered difference DataFrame.")
else:
    print("Filtered difference DataFrame ('diff_df') not found. Please ensure the previous cells have run and the filter matched some data.")

--- Filtering results for NS3 run: NS3 (cc:DCTCP, win:1, adapt:1, buf:1, size:1500) ---

--- G2 vs. NS3 (DCTCP) Time Difference Analysis ---


,Workload,Comm Type,Comm Size,NPU Tuples,NS3 Run,G2 Time,NS3 Time,Absolute Diff (NS3 - G2),Relative Diff (%)
1,T5_Base_split/0000_T5_Base_multiple_2_2_2_2_1_...,ALL_GATHER,16777216,"((0, 2), (1, 3), (4, 6), (5, 7))","NS3 (cc:DCTCP, win:1, adapt:1, buf:1, size:1500)","15,110,686,520.00","15,201,871,866.75","91,185,346.75",0.60
4,T5_Base_split/0001_T5_Base_multiple_2_2_2_2_1_...,REDUCE_SCATTER,33554432,"((0, 2), (1, 3), (4, 6), (5, 7))","NS3 (cc:DCTCP, win:1, adapt:1, buf:1, size:1500)","15,110,701,544.00","15,201,886,890.75","91,185,346.75",0.60
7,T5_Base_split/0002_T5_Base_multiple_2_2_2_2_1_...,REDUCE_SCATTER,33554432,"((0, 4), (1, 5), (2, 6), (3, 7))","NS3 (cc:DCTCP, win:1, adapt:1, buf:1, size:1500)","17,269,372,015.25","19,542,748,086.50","2,273,376,071.25",13.16
10,T5_Base_split/0003_T5_Base_multiple_2_2_2_2_1_...,ALL_GATHER,16777216,"((0, 4), (1, 5), (2, 6), (3, 7))","NS3 (cc:DCTCP, win:1, adapt:1, buf:1, size:1500)","17,269,356,991.25","19,542,733,062.50","2,273,376,071.25",13.16
13,T5_Base_split/0004_T5_Base_multiple_2_2_2_2_1_...,ALL_GATHER,3145728,"((0, 1), (2, 3), (4, 5), (6, 7))","NS3 (cc:DCTCP, win:1, adapt:1, buf:1, size:1500)","1,619,020,020.00","1,661,748,020.00","42,728,000.00",2.64
...,...,...,...,...,...,...,...,...,...
1254,T5_Base_split_4/0105_T5_Base_multiple_8_1_1_2_...,ALL_GATHER,1572864,"((8, 9, 10, 11, 12, 13, 14, 15),)","NS3 (cc:DCTCP, win:1, adapt:1, buf:1, size:1500)","5,666,570,080.00","6,731,522,008.50","1,064,951,928.50",18.79
1257,T5_Base_split_4/0106_T5_Base_multiple_8_1_1_2_...,REDUCE_SCATTER,12582912,"((8, 9, 10, 11, 12, 13, 14, 15),)","NS3 (cc:DCTCP, win:1, adapt:1, buf:1, size:1500)","5,666,579,929.00","6,734,640,934.75","1,068,061,005.75",18.85
1260,T5_Base_split_4/0107_T5_Base_multiple_8_1_1_2_...,ALL_REDUCE,12582912,"((0, 1, 2, 3, 4, 5, 6, 7),)","NS3 (cc:DCTCP, win:1, adapt:1, buf:1, size:1500)","11,333,149,999.00","13,365,713,398.00","2,032,563,399.00",17.93
1263,T5_Base_split_4/0108_T5_Base_multiple_8_1_1_2_...,ALL_REDUCE,12582912,"((8, 9, 10, 11, 12, 13, 14, 15),)","NS3 (cc:DCTCP, win:1, adapt:1, buf:1, size:1500)","11,333,149,999.00","13,626,905,100.62","2,293,755,101.62",20.24



--- Correlation Analysis (for DCTCP run) ---
Correlation with 'Relative Diff (%)':


,Relative Diff (%)
Relative Diff (%),1.000000
Absolute Diff (NS3 - G2),0.538899
NS3 Time,0.074511
comm_size_bytes,0.062515
G2 Time,0.007194
npu_count,-0.022108



--- Analyzing the Impact of Communication Type (for DCTCP run) ---

This plot shows how the percentage difference between NS3 (DCTCP) and G2 varies for each communication type.



This plot shows the distribution of G2 simulation times for each communication type.



This plot shows the distribution of NS3 (DCTCP) simulation times for each communication type.



--- Average Metrics by Communication Type (for DCTCP run) ---


,avg_g2_time,avg_ns3_time,avg_relative_diff,count
Comm Type,,,,
ALL_REDUCE,"9,841,925,381.37","10,736,813,852.62",9.96,148
REDUCE_SCATTER,"18,828,016,578.43","20,347,318,762.85",7.71,120
ALL_GATHER,"14,625,579,324.52","15,742,143,955.27",6.64,164
